# Can we predict 30-day readmission?
Data: `data.csv` (one row per hospital stay), dictionary in `columns.csv`. Target: `readmit_30d`.

Key design point: the same person appears in many rows (~8 stays each), so rows are **not independent**. All evaluation here uses cross-validation grouped by `person_key`, so a person never appears in both train and test. A naive random split is shown at the end for comparison.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve
from sklearn.inspection import permutation_importance
pd.set_option('display.width',140)
df=pd.read_csv('data.csv'); cols=pd.read_csv('columns.csv')
df.shape

## 1. Data checks

In [ ]:
print(df.isna().sum().sum(),'missing;',df.duplicated().sum(),'duplicate rows;',df.person_key.nunique(),'people')
print('Readmission rate: %.3f'%df.readmit_30d.mean())
print(df.groupby('person_key').size().describe())
df.describe().T

In [ ]:
# Is readmission clustered within people? (would make person-level info informative)
g=df.groupby('person_key').readmit_30d.agg(['mean','size'])
p=df.readmit_30d.mean()
print('Observed variance of per-person rate: %.4f'%g['mean'].var(), '| expected if independent: %.4f'%(p*(1-p)/g['size'].mean()))

## 2. Univariate look at each feature

In [ ]:
feats=['visit_number','birth_year','postcode_area','bmi','length_of_stay_days','n_medications','lab_a','lab_b']
rows=[(f, roc_auc_score(df.readmit_30d, df[f])) for f in feats]
rows.append(('sex==M', roc_auc_score(df.readmit_30d,(df.sex=='M').astype(int))))
print(pd.DataFrame(rows,columns=['feature','univariate AUC']).set_index('feature').round(3))
fig,ax=plt.subplots(2,4,figsize=(16,6))
for a,f in zip(ax.ravel(),feats):
    q=pd.qcut(df[f],10,duplicates='drop')
    r=df.groupby(q,observed=True).readmit_30d.mean()
    a.plot(range(len(r)),r.values,'o-'); a.axhline(p,color='grey',ls='--'); a.set_title(f); a.set_ylim(0,0.4)
plt.suptitle('Readmission rate by feature decile'); plt.tight_layout()

## 3. Models (grouped 5-fold CV, out-of-fold predictions)
`person_key` is an identifier and is **not** used as a feature.

In [ ]:
df['sex_M']=(df.sex=='M').astype(int)
X=df[feats+['sex_M']]; y=df.readmit_30d; groups=df.person_key
cv=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=0)
models={
 'baseline (prior)':DummyClassifier(strategy='prior'),
 'logistic regression':make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000)),
 'gradient boosting':HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=150,min_samples_leaf=50,random_state=0),
}
oof={}; res=[]
for n,m in models.items():
    pr=cross_val_predict(m,X,y,cv=cv,groups=groups,method='predict_proba')[:,1]; oof[n]=pr
    aucs=[roc_auc_score(y.iloc[te],pr[te]) for _,te in cv.split(X,y,groups)]
    res.append((n,roc_auc_score(y,pr),np.std(aucs),average_precision_score(y,pr),brier_score_loss(y,pr)))
pd.DataFrame(res,columns=['model','ROC AUC','AUC sd (folds)','PR AUC','Brier']).set_index('model').round(4)

In [ ]:
# Bootstrap-by-person CI for the AUC
rng=np.random.default_rng(0); ids=df.person_key.unique(); idx=df.groupby('person_key').indices
def ci(pr,B=300):
    out=[]
    for _ in range(B):
        s=np.concatenate([idx[i] for i in rng.choice(ids,len(ids))])
        out.append(roc_auc_score(y.values[s],pr[s]))
    return np.percentile(out,[2.5,97.5])
for n in ['logistic regression','gradient boosting']:
    print(n,'AUC 95%% CI: %.3f-%.3f'%tuple(ci(oof[n])))
plt.figure(figsize=(5,5))
for n,pr in oof.items():
    f,t,_=roc_curve(y,pr); plt.plot(f,t,label=n)
plt.plot([0,1],[0,1],'k:'); plt.legend(); plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('Out-of-fold ROC')

## 4. What drives the signal?

In [ ]:
lr=models['logistic regression'].fit(X,y)
print(pd.Series(lr[-1].coef_[0],index=X.columns).sort_values().round(3).to_string())
imp=np.zeros(X.shape[1])
for tr,te in cv.split(X,y,groups):
    m=models['gradient boosting'].fit(X.iloc[tr],y.iloc[tr])
    imp+=permutation_importance(m,X.iloc[te],y.iloc[te],scoring='roc_auc',n_repeats=5,random_state=0).importances_mean/5
pd.Series(imp,index=X.columns).sort_values().plot.barh(title='Permutation importance (drop in AUC, held-out)')

## 5. Calibration and usefulness

In [ ]:
pr=oof['logistic regression']
q=pd.qcut(pr,10,duplicates='drop')
print(pd.DataFrame({'pred':pr,'obs':y}).groupby(q,observed=True).mean().round(3))
top=pr>=np.quantile(pr,0.8)
print('Top 20%% risk: readmit rate %.3f vs overall %.3f (lift %.2fx)'%(y[top].mean(),y.mean(),y[top].mean()/y.mean()))

## 6. Sanity check: why grouping matters

In [ ]:
m=HistGradientBoostingClassifier(max_depth=6,max_iter=300,random_state=0)
Xn=X.assign(person_key=df.person_key)  # identifier as feature + random split: what NOT to do
skf=StratifiedKFold(5,shuffle=True,random_state=0)
for name,XX,splitter,kw in [('grouped, no id',X,cv,{'groups':groups}),
                            ('random split, no id',X,skf,{}),
                            ('random split, with person_key',Xn,skf,{})]:
    pr=cross_val_predict(m,XX,y,cv=splitter,method='predict_proba',**kw)[:,1]
    print('%-32s AUC %.3f'%(name,roc_auc_score(y,pr)))

## 7. Person-level history
Section 1 showed readmission is strongly clustered within people (per-person variance of the rate is ~3.5x what independence would give), so some people are persistently higher risk than the measured covariates explain. For a *new* stay, the outcomes of that person's **earlier** stays are known at prediction time, so we can legitimately use them. Assumption: `visit_number` orders stays chronologically. Features use only strictly earlier visits (no leakage of the current row's label).

In [ ]:
h=df.sort_values(['person_key','visit_number']).copy()
gp=h.groupby('person_key')
h['prior_n']=gp.cumcount()
h['prior_readmits']=gp.readmit_30d.cumsum()-h.readmit_30d
# smoothed prior rate (shrunk toward the global rate with k pseudo-visits)
k=5; h['prior_rate']=(h.prior_readmits+k*p)/(h.prior_n+k)
for c in ['length_of_stay_days','n_medications','lab_a']:
    h['prior_mean_'+c]=((gp[c].cumsum()-h[c])/h.prior_n.replace(0,np.nan))
h=h.sort_index()
hist=['prior_n','prior_readmits','prior_rate']
X2=h[feats+['sex_M']+hist]
res=[]
for n,m in {'logistic regression':models['logistic regression'],'gradient boosting':models['gradient boosting']}.items():
    pr=cross_val_predict(m,X2,y,cv=cv,groups=groups,method='predict_proba')[:,1]
    res.append((n+' + history',roc_auc_score(y,pr),average_precision_score(y,pr),brier_score_loss(y,pr)))
    oof[n+' + history']=pr
pd.DataFrame(res,columns=['model','ROC AUC','PR AUC','Brier']).set_index('model').round(4)

In [ ]:
# Performance on first stays (no history available) vs later stays, with the history model
pr=oof['logistic regression + history']; first=(h.prior_n==0).values
for name,mask in [('first stay (no history)',first),('stays with history',~first)]:
    print('%-26s n=%5d  AUC without history %.3f | with history %.3f'%(name,mask.sum(),
          roc_auc_score(y[mask],oof['logistic regression'][mask]),roc_auc_score(y[mask],pr[mask])))

## Conclusion
(Figures from a run of these cells; re-run to refresh.)

- **Stay-level features alone: weak but real signal.** Grouped-CV AUC ≈ 0.65 (95% CI ≈ 0.63-0.66) vs 0.50 for the baseline. Drivers are `length_of_stay_days`, `lab_a` and `n_medications` (higher → more readmission); age, sex, BMI, `lab_b`, postcode and visit number carry ~nothing. Logistic regression is at least as good as gradient boosting, so the signal is roughly linear. Calibration is good; the top-20% risk group is readmitted ~34% of the time vs 20% overall (1.7x lift).
- **Prior outcomes for the same person: much stronger.** Adding the person's earlier readmission history lifts AUC to ≈ 0.82 overall and ≈ 0.84 on stays that have history. A first stay has no history, and there AUC stays ≈ 0.62.
- **Evaluation caveat.** Rows from one person are not independent. A random row split inflates AUC (≈ 0.75 even without the ID, because the other columns partly identify a person). Use person-grouped splits.
- **Assumptions to confirm.** `visit_number` is chronological, and prior-stay outcomes are available when predicting a new stay. If predictions must be made without linked history, expect ≈ 0.65. This is retrospective, single-dataset evidence, with no external validation.